In [3]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Load all CSV files from the data folder and merge them
data_dir = "/content/drive/MyDrive/flowwatch/data"
csv_files = sorted(f for f in os.listdir(data_dir) if f.endswith(".csv"))
dfs = [pd.read_csv(os.path.join(data_dir, f)) for f in csv_files]
print(f"Loaded {len(dfs)} file(s): {csv_files}")

df = pd.concat(dfs, axis=0, ignore_index=True)
print(f"Merged shape: {df.shape}")

Loaded 8 file(s): ['Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 'Monday-WorkingHours.pcap_ISCX.csv', 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 'Wednesday-workingHours.pcap_ISCX.csv']
Merged shape: (2830743, 79)


In [6]:
# Strip whitespace from column names
df.columns = df.columns.str.strip()

In [7]:
# Drop duplicate rows
print(f"Rows before: {len(df)}")
df.drop_duplicates(inplace=True)
print(f"Rows after dropping duplicates: {len(df)}")

Rows before: 2830743
Rows after dropping duplicates: 2522362


In [8]:
# Replace infinite values with NaN, then drop all rows with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
print(f"Rows after dropping inf/NaN: {len(df)}")

Rows after dropping inf/NaN: 2520798


In [9]:
# Encode the Label column
le = LabelEncoder()
df["Label"] = le.fit_transform(df["Label"])

# Print label mapping
for i, label in enumerate(le.classes_):
    print(f"  {label} -> {i}")

  BENIGN -> 0
  Bot -> 1
  DDoS -> 2
  DoS GoldenEye -> 3
  DoS Hulk -> 4
  DoS Slowhttptest -> 5
  DoS slowloris -> 6
  FTP-Patator -> 7
  Heartbleed -> 8
  Infiltration -> 9
  PortScan -> 10
  SSH-Patator -> 11
  Web Attack � Brute Force -> 12
  Web Attack � Sql Injection -> 13
  Web Attack � XSS -> 14


In [12]:
# Save cleaned dataset
df.to_csv("/content/drive/MyDrive/flowwatch/data/cleaned.csv", index=False)
print(f"Saved to data/cleaned.csv — shape: {df.shape}")

Saved to data/cleaned.csv — shape: (2520798, 79)


In [ ]:
# Upload the cleaned dataset to the HuggingFace dataset repo (versioned artifact)
%pip install huggingface_hub -q
from huggingface_hub import HfApi
from google.colab import userdata

HF_REPO = "valthe/flowwatch"
api = HfApi(token=userdata.get("HF_TOKEN"))
api.create_repo(HF_REPO, repo_type="dataset", exist_ok=True)
api.upload_file(
    path_or_fileobj="/content/drive/MyDrive/flowwatch/data/cleaned.csv",
    path_in_repo="cleaned.csv",
    repo_id=HF_REPO,
    repo_type="dataset",
)
print(f"Uploaded cleaned.csv to https://huggingface.co/datasets/{HF_REPO}")